# 05 · No fine-tuning: model comparison with retrieved examples

No training at all. For every question, the 4 most similar **training** questions are shown to the model as solved examples (their evidence rows, unit rows and correct 4-line answer). The model learns the dataset's answer-type and unit conventions from them. Python still computes every number.

Same 200 dev questions as notebook 01 (where Qwen2.5-7B with 2 fixed examples scored **42.0 EM**), so the numbers are directly comparable.

Run the cells **one at a time**. Each model cell takes roughly 20–60 min on a T4 and resumes after a disconnect (re-run setup, the helper cell, then the same model cell).

In [ ]:
# --- Setup: on Colab, gets the latest code + data from GitHub; also works on a local GPU ---
import os, sys, subprocess, shutil, random
IN_COLAB = os.path.exists("/content")
if IN_COLAB:
    ROOT = "/content/finnn"
    shutil.rmtree(ROOT, ignore_errors=True)               # always fetch the latest code
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/praseeda-cse/finnn.git", ROOT], check=True)
    try:                                                  # save results on Drive so they survive disconnects
        from google.colab import drive
        drive.mount("/content/drive")
        OUT = "/content/drive/MyDrive/finsight/runs"
    except Exception as e:
        print("Drive not mounted (", e, ") - results are saved in /content/runs and are LOST if Colab disconnects")
        OUT = "/content/runs"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "peft", "bitsandbytes", "accelerate"], check=True)
else:
    ROOT = os.path.abspath("..")
    OUT = os.path.join(ROOT, "runs")
os.makedirs(OUT, exist_ok=True)
sys.path.insert(0, ROOT)

import torch
from finsight.data import load_split
from finsight.evaluate import score, print_report
from finsight.infer import load_model, predict
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - select a GPU runtime!")

docs_train, q_train = load_split(f"{ROOT}/data/tatdqa_train.json.gz")
docs_dev, q_dev = load_split(f"{ROOT}/data/tatdqa_dev.json.gz")
docs_test, q_test = load_split(f"{ROOT}/data/tatdqa_test.json.gz")
def sample(questions, n, seed=0):
    return random.Random(seed).sample(questions, n) if n < len(questions) else questions
def latest_adapter(run):
    """The finished adapter, or the newest checkpoint if training stopped midway."""
    import glob
    if os.path.exists(f"{run}/final"):
        return f"{run}/final"
    ckpts = sorted(glob.glob(f"{run}/checkpoint-*"), key=lambda p: int(p.rsplit("-", 1)[1]))
    assert ckpts, f"No adapter in {run} - run notebook 02 (training) first"
    print("NOTE: training has not finished; using", ckpts[-1])
    return ckpts[-1]
print(len(q_train), len(q_dev), len(q_test), "questions")

In [ ]:
import gc, json
from finsight.examples import ExampleRetriever

N = 200
eval_qs = sample(q_dev, N)                 # same sample as notebook 01 (seed 0)
retriever = ExampleRetriever(docs_train, q_train)
examples_fn = lambda q: retriever(q, k=4)
RESULTS = {}
base_file = f"{OUT}/zeroshot_Qwen2.5-7B-Instruct_dev{N}.jsonl"
if os.path.exists(base_file):
    base = {json.loads(l)["uid"]: json.loads(l) for l in open(base_file)}
    RESULTS["Qwen2.5-7B, 2 fixed examples (notebook 01)"] = score(eval_qs, base)

def run(model_id, batch_size=4, template_kwargs=None, label=None):
    name = model_id.split("/")[-1]
    model, tok = load_model(model_id)
    preds = predict(model, tok, docs_dev, eval_qs, f"{OUT}/noft_{name}_retrieved4_dev{N}.jsonl",
                    batch_size=batch_size, examples_fn=examples_fn, template_kwargs=template_kwargs)
    res = score(eval_qs, preds)
    RESULTS[label or f"{name}, 4 retrieved examples"] = res
    print_report(res, label or name)
    del model; gc.collect(); torch.cuda.empty_cache()
    return preds

## Model A · Qwen2.5-7B-Instruct + retrieved examples
Same model as notebook 01, so this shows what the retrieved examples alone add.

In [ ]:
preds_a = run("Qwen/Qwen2.5-7B-Instruct", batch_size=4)

## Model B · Qwen3-8B (thinking turned off)

In [ ]:
preds_b = run("Qwen/Qwen3-8B", batch_size=4, template_kwargs={"enable_thinking": False})

## Model C · Qwen2.5-14B-Instruct (largest that fits a free T4 in 4-bit; slowest)

In [ ]:
preds_c = run("Qwen/Qwen2.5-14B-Instruct", batch_size=2)

## Comparison table

In [ ]:
rows = []
for label, r in RESULTS.items():
    rows.append([label, f"{r['EM']:.1f}", f"{r['F1']:.1f}", f"{r['scale_acc']:.1f}",
                 f"{r.get('arithmetic_EM', 0):.1f}", f"{r.get('span_EM', 0):.1f}",
                 f"{r['evidence_recall']:.0f}/{r['evidence_precision']:.0f}"])
head = ["setup", "EM", "F1", "unit acc", "arith EM", "span EM", "evid R/P"]
w = [max(len(str(x)) for x in col) for col in zip(head, *rows)]
for row in [head] + rows:
    print("  ".join(str(x).ljust(n) for x, n in zip(row, w)))
json.dump({k: {m: v for m, v in r.items() if m != "rows"} for k, r in RESULTS.items()},
          open(f"{OUT}/noft_comparison_dev{N}.json", "w"), indent=1)